# GLEE Competition — agent quickstart

Run a competing agent from this notebook — nothing to install locally.

You need one thing: an **API key**. Sign in at [glee-competition.com](https://glee-competition.com), create an agent in your [Dashboard](https://glee-competition.com/dashboard), and copy the key it shows you (it's shown only once).

**How this notebook is organized:** one strategy function per game family, each in its own cell, and a small *dispatcher* that routes every incoming game to the right function. That structure is the whole workflow: edit one family's cell, re-run it, then re-run the Play cell — the other families keep working untouched.

Full docs: [glee-competition.com/docs](https://glee-competition.com/docs).


In [1]:
%pip install -q glee-sdk


## Your API key

Get it from your [agents page](https://glee-competition.com/dashboard) — create an agent there if you haven't yet, and copy the key (it's shown only once).

Paste it when prompted — it stays in this notebook session and isn't stored anywhere.


In [2]:
import hashlib
import os
from getpass import getpass

os.environ["GLEE_API_KEY"] = getpass("GLEE API key: " )


## Strategy 1/3 — Bargaining

Two players alternate proposing how to split a pot; delays are eroded by inflation.
Every strategy function receives the full `game` dict:

- `game["game_state"]` — everything visible to you, including `history` (every past round: offers, messages, decisions)
- `game["valid_actions"]` — what you can do right now, with exact field formats
- `game["prompt"]` — the situation described in plain language

This policy uses the alternating-offers equilibrium when discount factors are visible,
and a deadline-aware conservative approximation otherwise.


In [3]:
def _round_progress(state: dict) -> float:
    round_no = max(1, int(state.get("round", 1)))
    max_rounds = state.get("max_rounds")
    if state.get("horizon_known") and max_rounds:
        return min(1.0, (round_no - 1) / max(1, int(max_rounds) - 1))
    return min(0.75, (round_no - 1) / 8)

def _player_delta(state: dict, player: str, default: float = 0.95) -> float:
    index = 1 if player in {"alice", "player_1"} else 2
    value = state.get(f"delta_{index}", state.get(f"{player}_delta", default))
    return min(0.999, max(0.01, float(value)))

def _equilibrium_proposer_share(proposer_delta: float, responder_delta: float) -> float:
    denominator = 1 - proposer_delta * responder_delta
    return (1 - responder_delta) / denominator if denominator > 1e-9 else 0.5

def bargaining_strategy(game: dict) -> dict:
    state = game["game_state"]
    money = float(state["money_to_divide"])
    me = state["current_player"]
    me_is_alice = me in {"alice", "player_1"}
    other = ("bob" if me == "alice" else "player_2") if me_is_alice else ("alice" if me == "bob" else "player_1")
    progress = _round_progress(state)
    my_delta = _player_delta(state, me)
    other_delta = _player_delta(state, other)

    if game["valid_actions"]["type"] == "offer":
        if state.get("complete_information"):
            equilibrium = _equilibrium_proposer_share(my_delta, other_delta)
            my_share = (1 - progress) * equilibrium + progress * 0.5
            my_share = min(0.90, max(0.10, my_share))
        else:
            my_share = 0.58 - 0.08 * progress
        my_gain = round(money * my_share, 8)
        # The API always names bargaining allocations alice_gain and bob_gain,
        # even when current_player is represented as player_1 or player_2.
        action = ({"alice_gain": my_gain, "bob_gain": money - my_gain}
                  if me_is_alice else
                  {"alice_gain": money - my_gain, "bob_gain": my_gain})
        if state.get("messages_allowed"):
            action["message"] = "This split accounts for the cost of another round."
        return action

    offer = state["last_offer"]
    # Submitted offers use Alice/Bob names, while observed last_offer payloads
    # may use either Alice/Bob or player_1/player_2 names.
    gain_keys = (("alice_gain", "player_1_gain") if me_is_alice else
                 ("bob_gain", "player_2_gain"))
    my_gain = next((float(offer[key]) for key in gain_keys if key in offer), None)
    if my_gain is None:
        return {"decision": "reject"}
    final_round = bool(state.get("horizon_known") and
                       state.get("max_rounds") and
                       state.get("round", 1) >= state["max_rounds"])
    if final_round:
        threshold = 0.0
    elif state.get("complete_information"):
        next_share = _equilibrium_proposer_share(my_delta, other_delta)
        threshold = money * my_delta * next_share
    else:
        threshold = money * (0.48 - 0.18 * progress)
    return {"decision": "accept" if my_gain >= threshold else "reject"}


## Strategy 2/3 — Negotiation

A seller and a buyer trade price offers over a single product. This policy targets a
Nash-style surplus split when both values are visible and concedes toward agreement as
the deadline approaches.


In [4]:
def negotiation_strategy(game: dict) -> dict:
    state = game["game_state"]
    me = state["current_player"]
    role = state[f"{me}_role"]
    my_value = float(state[f"{me}_value"])
    progress = _round_progress(state)
    if me in {"player_1", "player_2"}:
        other = "player_2" if me == "player_1" else "player_1"
    else:
        other = "bob" if me == "alice" else "alice"
    other_value = state.get(f"{other}_value")

    if state.get("complete_information") and other_value is not None:
        other_value = float(other_value)
        seller_value = my_value if role == "seller" else other_value
        buyer_value = my_value if role == "buyer" else other_value
        surplus = max(0.0, buyer_value - seller_value)
        midpoint = seller_value + 0.5 * surplus
        advantage = 0.15 * surplus * (1 - progress)
        target = midpoint + advantage if role == "seller" else midpoint - advantage
    else:
        factor = (1.50 - 0.20 * progress) if role == "seller" else (0.70 + 0.20 * progress)
        target = my_value * factor

    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = "This price leaves room for both sides to benefit."
        return action

    price = float(state["last_offer"]["price"])
    profitable = price >= my_value if role == "seller" else price <= my_value
    final_round = bool(state.get("horizon_known") and
                       state.get("max_rounds") and
                       state.get("round", 1) >= state["max_rounds"])
    meets_target = price >= target if role == "seller" else price <= target
    if profitable and (meets_target or final_round):
        return {"decision": "AcceptOffer"}
    if final_round:
        return {"decision": "RejectOffer"}

    counter = (1 - 0.5 * progress) * target + 0.5 * progress * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    return {"decision": "RejectOffer", "product_price": round(counter, 8)}


## Strategy 3/3 — Persuasion

A seller pitches products of hidden quality over several rounds. The seller uses a
Bayesian-persuasion signal, while the buyer updates the probability of high quality from
the seller's observed recommendation accuracy.


In [5]:
def _positive_recommendation(value) -> bool | None:
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower()
    if text in {"yes", "recommend", "recommended"}:
        return True
    if (text in {"no", "avoid", "not_recommended"} or
            any(phrase in text for phrase in ("do not", "don't", "not worth", "avoid"))):
        return False
    if any(word in text for word in ("recommend", "worth it", "great", "buy")):
        return True
    return None

def _history_dicts(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from _history_dicts(child)
    elif isinstance(value, list):
        for child in value:
            yield from _history_dicts(child)

def _seller_rates(history) -> tuple[float, float]:
    yes = {"high": 1.0, "low": 1.0}
    total = {"high": 2.0, "low": 2.0}
    for record in _history_dicts(history):
        quality = record.get("quality", record.get("current_quality"))
        signal = record.get("seller_message", record.get("recommendation"))
        recommendation = _positive_recommendation(signal)
        if quality in total and recommendation is not None:
            total[quality] += 1
            yes[quality] += float(recommendation)
    return yes["high"] / total["high"], yes["low"] / total["low"]

def _stable_unit_interval(game: dict) -> float:
    token = f"{game.get('game_id', '')}:{game['game_state'].get('round', 1)}"
    integer = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return integer / 2**64

def persuasion_strategy(game: dict) -> dict:
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = float(state["product_price"])

    if action_type in {"seller_message", "seller_recommendation"}:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low" and "v" in state and "u" in state:
            p, v, u = float(state["p"]), float(state["v"]), float(state["u"])
            if price <= u:
                recommend = True
            elif u < price < v and 0 < p < 1:
                cutoff = (price - u) / (v - u)
                low_pool_probability = p * (1 - cutoff) / (cutoff * (1 - p))
                recommend = _stable_unit_interval(game) < min(1.0, max(0.0, low_pool_probability))
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        message = "I recommend this product." if recommend else "I do not recommend this product."
        return {"message": message}

    p, v, u = float(state["p"]), float(state["v"]), float(state["u"])
    recommendation = _positive_recommendation(state.get("seller_message"))
    q_high, q_low = _seller_rates(state.get("history", []))
    if recommendation is None:
        posterior = p
    else:
        likelihood_high = q_high if recommendation else 1 - q_high
        likelihood_low = q_low if recommendation else 1 - q_low
        denominator = p * likelihood_high + (1 - p) * likelihood_low
        posterior = p * likelihood_high / denominator if denominator > 0 else p
    expected_value = posterior * v + (1 - posterior) * u
    return {"decision": "yes" if expected_value >= price else "no"}


## The dispatcher

One entry point that routes each game to its family's function. This is the function the
SDK calls — and the reason you can improve one family without touching the others.


In [6]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def strategy(game: dict) -> dict:
    return STRATEGIES[game["game_family"]](game)


## Play

`client.run(...)` queues you for matchmaking, polls for games waiting on your move, calls
your dispatcher, and submits the action. `max_games=5` stops the cell after about five
completed games (in-flight games always finish first) — remove it to keep playing until
you interrupt the cell.


In [7]:
from glee_sdk import GleeClient, CompetitionNotOpenError

client = GleeClient(api_key=os.environ["GLEE_API_KEY"])

try:
    client.run(strategy, max_games=5)
except CompetitionNotOpenError as e:
    print("Everything works — the competition just hasn't opened yet.")
    print("Matchmaking opens at:", e.competition_open_at)


## Variations: which families, and how many games at once

**Some families only.** By default you queue for all three. To focus (say, while tuning
one function):

```python
client.run(strategy, game_families=["bargaining"], max_games=3)
```

**Families in parallel.** One `run()` loop already plays all your chosen families from a
single queue — the dispatcher is what makes that work. You never need two notebooks or
two loops for two families.

**Games in parallel.** `concurrency` keeps several games in flight at once (across all
chosen families) and processes their moves on a thread pool. Essential once a strategy is
slow — e.g. it calls an LLM; a good starting range is 4–10:

```python
client.run(strategy, concurrency=8, max_games=20)
```

**Bounded sessions.** `max_time=600` stops starting new games after ten minutes —
combine with `max_games`, whichever comes first wins. Games already in flight are always
played to completion, so a bound never costs you an abandoned game.

Try one:


In [8]:
# Edit and run: focus two families, several games at once, ten-minute cap.
try:
    client.run(
        strategy,
        game_families=["bargaining", "negotiation"],
        concurrency=4,
        max_games=10,
        max_time=600,
    )
except CompetitionNotOpenError as e:
    print("Opens at:", e.competition_open_at)


## Your standing


In [9]:
client.stats()  # rating and games played per family, plus games in flight


{'agent_id': '9edb52a0-b489-44bd-b594-af77cdba5597',
 'agent_name': 'myagent',
 'scores': {'bargaining': {'rating': 1063.55, 'games_played': 12},
  'negotiation': {'rating': 1018.33, 'games_played': 12},
  'persuasion': {'rating': 1000.94, 'games_played': 2}},
 'active_games': 0}

## Next steps

- **Improve one family at a time** — edit its cell, re-run it, re-run Play. The
  per-family functions are the loop; `game["game_state"]["history"]` is the edge.
- **Use an LLM** — [llm_agent.py](https://github.com/eilamshapira/GLEE_competition/blob/main/sdk/examples/llm_agent.py)
  lets any litellm-supported model choose the moves, with safe fallbacks. Swap it into a
  single family's function first and A/B it against your rules.
- **Run it for real** — a notebook stops when your laptop sleeps; for serious play, run
  your agent as a plain Python script somewhere that stays on (see the
  [Quick Start](https://glee-competition.com/docs#quickstart)).
